In [1]:
# MONTO VOLUME ONE DRIVE (qui carico i video con id e sampled)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# 1. Rimuoviamo la versione AMD ROCm errata
!pip uninstall -y torch torchvision torchaudio flash-attn
# 2. Installiamo PyTorch 2.5.1 nativo per NVIDIA CUDA 12.4 (Perfetto per A100 su Colab)
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
# 3. Installiamo FlashAttention pre-compilato (Si installerà in meno di 20 secondi senza compilare!)
!pip install flash-attn --no-build-isolation --upgrade

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 133.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 112.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 151.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:0000:0100:

In [3]:
# 1. Aggiorna i pacchetti del sistema Colab e installa FFmpeg nativo
!apt-get update && apt-get install -y ffmpeg
# 2. Installa Transformers e Accelerate nelle versioni che hai scelto
!pip install transformers==4.46.3 accelerate==1.0.1
# 3. Installa i wrapper Python per la gestione dei video
!pip install decord ffmpeg-python imageio opencv-python
# 4. Clona il repo VideoLLaMA2 e loca rica su one Drive cosi che ce lo abbiamo a disposizione durante il processing
import os
if not os.path.exists("/content/VideoLLaMA2"):
    !git clone https://github.com/DAMO-NLP-SG/VideoLLaMA2.git /content/VideoLLaMA2
import sys
if "/content/VideoLLaMA2" not in sys.path:
    sys.path.insert(0, "/content/VideoLLaMA2")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,701 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,605 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,309 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 P

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
model_path   = "DAMO-NLP-SG/VideoLLaMA3-7B"
model_path_2 = "DAMO-NLP-SG/VideoLLaMA2-7B"

# --- VideoLLaMA3: caricamento standard via Transformers (architettura registrata) ---
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    device_map={"": device},
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

# --- VideoLLaMA2: caricamento tramite il package ufficiale (architettura videollama2_mistral) ---
# AutoModelForCausalLM non riconosce 'videollama2_mistral' senza il package dedicato
model2, processor2, tokenizer2 = videollama2_model_init(
    model_path_2,
    device_map={"": device},
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_videollama3.py: 0.00B [00:00, ?B/s]

configuration_videollama3_encoder.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/DAMO-NLP-SG/VideoLLaMA3-7B:
- configuration_videollama3_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

da qui in poi facci inferenza sui video e sulle annotation e calcolo le prime statistiche


In [ ]:

import json
import pandas as pd

# Percorso del JSON delle annotation
ANNOTATION_PATH = "/content/drive/MyDrive/PROGETTO DEEP LEARNING/nutrition_video_nutrition_estimation.json"
# Cartella dove si trovano i video campionati su Colab
VIDEO_DIR = "/content/drive/MyDrive/PROGETTO DEEP LEARNING/videos_sampled/"

with open(ANNOTATION_PATH, "r") as f:
    annotations = json.load(f)

print(f" Caricate {len(annotations)} annotation")



In [7]:
import re
import os

def build_prompt(question, choices):
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])
    return (
        f"Watch the video carefully and answer the following multiple-choice question.\n\n"
        f"Question: {question}\n\n"
        f"Choices:\n{choices_text}\n\n"
        f"Reply with ONLY the number (0-{len(choices)-1}) corresponding to the correct answer."
    )

def parse_answer(response_text, num_choices):
    matches = re.findall(r'\b([0-9])\b', response_text)
    for m in reversed(matches):
        idx = int(m)
        if 0 <= idx < num_choices:
            return idx
    return -1


def inference_on_videos(model, annotations, processor=None, tokenizer=None, model_type="videollama3"):
    """
    model_type: "videollama3" (default) oppure "videollama2"
    - videollama3: usa processor(conversation=...) + model.generate()
    - videollama2: usa mm_infer(processor, model, ..., tokenizer=tokenizer)
    """
    results = []

    for key, item in annotations.items():
        video_id   = item["inputs"]["video 1"]["id"]
        question   = item["question"]
        choices    = item["choices"]
        correct    = item["correct_idx"]
        video_path = f"{VIDEO_DIR}{video_id}_sampled.mp4"

        print(f"▶ {key} | video: {video_id}")

        try:
            if not os.path.exists(video_path):
                raise FileNotFoundError(f"Video non trovato: {video_path}")

            if model_type == "videollama2":
                # ── API VideoLLaMA2 ─────────────────────────────────────────
                # mm_infer(image_or_video, instruct, model, tokenizer, modal, **kwargs)
                # Il 1° argomento è il TENSORE già preprocessato, non il processor
                from videollama2 import mm_infer
                video_tensor = processor['video'](video_path)
                response = mm_infer(
                    video_tensor,
                    build_prompt(question, choices),
                    model,
                    tokenizer,
                    modal="video",
                    do_sample=False,
                    max_new_tokens=64,
                )

            else:
                # ── API VideoLLaMA3: processor + model.generate ─────────────
                SYSTEM_PROMPT = """
                You are an expert nutritional analyst watching a cooking video.
                Your task: given the video frames and a multiple-choice question,
                select the single best answer among 4 options.

                ## Task Breakdown
                1. **Observe** the ingredients shown, cooking techniques, and final dish.
                2. **Analyze** nutritional aspects: macro/micronutrients, portions, balance.
                3. **Evaluate** each of the 4 answer options against what you see.
                4. **Select** the most accurate answer and output ONLY its label (0/1/2/3).

                ## Output Format
                Respond with exactly one of these labels, nothing else:
                - 0
                - 1
                - 2
                - 3

                ## Constraints
                - Base your answer ONLY on visible evidence in the video.
                - If no answer is clearly correct, pick the least incorrect one.
                - Do NOT hallucinate ingredients or techniques not present.
                - Prioritize accuracy over guessing.
                """
                conversation = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "video",
                                "video": {
                                    "video_path": video_path,
                                    "fps": 1,
                                    "max_frames": 128,
                                }
                            },
                            {"type": "text", "text": build_prompt(question, choices)},
                        ]
                    },
                ]

                inputs = processor(
                    conversation=conversation,
                    add_system_prompt=True,
                    add_generation_prompt=True,
                    return_tensors="pt",
                )

                # Guardia esplicita: il processor restituisce None se il video non è leggibile
                if inputs.get("input_ids") is None:
                    raise ValueError(
                        "Il processor ha restituito input_ids=None. "
                        "Il file video potrebbe essere corrotto o in un formato non supportato."
                    )

                inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
                if "pixel_values" in inputs:
                    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

                output_ids = model.generate(**inputs, max_new_tokens=64)
                response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

            predicted  = parse_answer(response, len(choices))
            is_correct = (predicted == correct)

            print(f"   Risposta grezza: {repr(response[-80:])}")
            print(f"   Predetto: {predicted} | Corretto: {correct} | {'✅' if is_correct else '❌'}\n")

        except Exception as e:
            response   = f"ERRORE: {e}"
            predicted  = -1
            is_correct = False
            print(f"   ⚠️ Errore: {e}\n")

        q_type = question.split("highest")[-1].strip().rstrip("?").strip() if "highest" in question else "unknown"

        results.append({
            "key":           key,
            "video_id":      video_id,
            "question":      question,
            "q_type":        q_type,
            "correct_idx":   correct,
            "predicted_idx": predicted,
            "is_correct":    is_correct,
            "raw_response":  response,
        })

    df = pd.DataFrame(results)
    print(f"\n🏁 Inferenza completata su {len(df)} campioni")
    return df


In [ ]:
results1 = inference_on_videos(model, annotations, processor=processor, model_type="videollama3")

In [ ]:

# ── Analisi delle performance ──────────────────────────────────────────────────

# Converte la lista di risultati in DataFrame (se non lo è già)
# if not isinstance(results1, pd.DataFrame):
#     df = pd.DataFrame(df)
df = results1
total     = len(df)
correct   = df["is_correct"].sum()
accuracy  = correct / total * 100
random_baseline = 100 / 5  # 5 scelte sempre

print("=" * 50)
print(f"  ACCURACY GLOBALE : {accuracy:.1f}%  ({correct}/{total})")
print(f"  Baseline random  : {random_baseline:.1f}%")
print("=" * 50)

# Accuracy per tipo di domanda (carbs / fat / protein / calories)
print("\n📊 Accuracy per tipo di domanda:")
acc_by_type = (
    df.groupby("q_type")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_type["accuracy_%"] = (acc_by_type["corrette"] / acc_by_type["totali"] * 100).round(1)
print(acc_by_type.to_string())

# Accuracy per video
print("\n📹 Accuracy per video:")
acc_by_video = (
    df.groupby("video_id")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_video["accuracy_%"] = (acc_by_video["corrette"] / acc_by_video["totali"] * 100).round(1)
print(acc_by_video.to_string())

# Salva i risultati su CSV nel Drive
output_csv = "/content/drive/MyDrive/inference_results.csv"
df.to_csv(output_csv, index=False)
print(f"\n💾 Risultati salvati in: {output_csv}")


In [ ]:
df = inference_on_videos(model2, annotations, processor=processor2, tokenizer=tokenizer2, model_type="videollama2")

In [ ]:

# ── Analisi delle performance ──────────────────────────────────────────────────

# Converte la lista di risultati in DataFrame (se non lo è già)
if not isinstance(df, pd.DataFrame):
    df = pd.DataFrame(df)

total     = len(df)
correct   = df["is_correct"].sum()
accuracy  = correct / total * 100
random_baseline = 100 / 5  # 5 scelte sempre

print("=" * 50)
print(f"  ACCURACY GLOBALE : {accuracy:.1f}%  ({correct}/{total})")
print(f"  Baseline random  : {random_baseline:.1f}%")
print("=" * 50)

# Accuracy per tipo di domanda (carbs / fat / protein / calories)
print("\n📊 Accuracy per tipo di domanda:")
acc_by_type = (
    df.groupby("q_type")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_type["accuracy_%"] = (acc_by_type["corrette"] / acc_by_type["totali"] * 100).round(1)
print(acc_by_type.to_string())

# Accuracy per video
print("\n📹 Accuracy per video:")
acc_by_video = (
    df.groupby("video_id")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_video["accuracy_%"] = (acc_by_video["corrette"] / acc_by_video["totali"] * 100).round(1)
print(acc_by_video.to_string())

# Salva i risultati su CSV nel Drive
output_csv = "/content/drive/MyDrive/inference_results.csv"
df.to_csv(output_csv, index=False)
print(f"\n💾 Risultati salvati in: {output_csv}")


In [5]:

import json
import pandas as pd

# Percorso del JSON delle annotation
ANNOTATION_PATH = "/content/drive/MyDrive/PROGETTO DEEP LEARNING/nutrition_nutrition_change.json"
# Cartella dove si trovano i video campionati su Colab
VIDEO_DIR = "/content/drive/MyDrive/PROGETTO DEEP LEARNING/videos_2_sampled/"

with open(ANNOTATION_PATH, "r") as f:
    annotations = json.load(f)

print(f" Caricate {len(annotations)} annotation")



 Caricate 50 annotation


In [8]:
results1 = inference_on_videos(model, annotations, processor=processor, model_type="videollama3")

▶ nutrition_nutrition_change_0 | video: P05-20240425-171455
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 0 | ❌

▶ nutrition_nutrition_change_1 | video: P03-20240217-131219
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 2 | ✅

▶ nutrition_nutrition_change_2 | video: P07-20240529-191007
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 1 | ❌

▶ nutrition_nutrition_change_3 | video: P04-20240414-065311
   Risposta grezza: '1'
   Predetto: 1 | Corretto: 4 | ❌

▶ nutrition_nutrition_change_4 | video: P08-20240614-085000
   Risposta grezza: '1'
   Predetto: 1 | Corretto: 3 | ❌

▶ nutrition_nutrition_change_5 | video: P09-20240624-160737
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 1 | ❌

▶ nutrition_nutrition_change_6 | video: P01-20240203-130505
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_nutrition_change_7 | video: P06-20240510-115307
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 3 | ❌

▶ nutrition_nutrition_change_8 | video: P06-20240510-100

In [10]:

# ── Analisi delle performance ──────────────────────────────────────────────────
df = results1

# Converte la lista di risultati in DataFrame (se non lo è già)
if not isinstance(df, pd.DataFrame):
    df = pd.DataFrame(df)

total     = len(df)
correct   = df["is_correct"].sum()
accuracy  = correct / total * 100
random_baseline = 100 / 5  # 5 scelte sempre

print("=" * 50)
print(f"  ACCURACY GLOBALE : {accuracy:.1f}%  ({correct}/{total})")
print(f"  Baseline random  : {random_baseline:.1f}%")
print("=" * 50)

# Accuracy per tipo di domanda (carbs / fat / protein / calories)
print("\n📊 Accuracy per tipo di domanda:")
acc_by_type = (
    df.groupby("q_type")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_type["accuracy_%"] = (acc_by_type["corrette"] / acc_by_type["totali"] * 100).round(1)
print(acc_by_type.to_string())

# Accuracy per video
print("\n📹 Accuracy per video:")
acc_by_video = (
    df.groupby("video_id")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_video["accuracy_%"] = (acc_by_video["corrette"] / acc_by_video["totali"] * 100).round(1)
print(acc_by_video.to_string())

# Salva i risultati su CSV nel Drive
output_csv = "/content/drive/MyDrive/inference_results.csv"
df.to_csv(output_csv, index=False)
print(f"\n💾 Risultati salvati in: {output_csv}")


  ACCURACY GLOBALE : 12.0%  (6/50)
  Baseline random  : 20.0%

📊 Accuracy per tipo di domanda:
         corrette  totali  accuracy_%
q_type                               
unknown         6      50        12.0

📹 Accuracy per video:
                     corrette  totali  accuracy_%
video_id                                         
P01-20240203-130505         1       2        50.0
P01-20240203-135502         0       3         0.0
P01-20240204-121042         0       2         0.0
P03-20240217-131219         1       2        50.0
P03-20240217-192543         0       5         0.0
P04-20240413-151722         1       2        50.0
P04-20240414-065311         0       2         0.0
P04-20240414-165333         0       1         0.0
P04-20240415-181212         0       1         0.0
P05-20240423-170021         0       1         0.0
P05-20240423-172243         0       1         0.0
P05-20240423-180529         0       2         0.0
P05-20240425-171455         0       1         0.0
P05-20240427-14552